# Activation Geometry

Geometric properties of activation spaces: manifold dimensionality, representation similarity across inputs, clustering, curvature, and norm distributions.

**References:**
- Ansuini et al. (2019) "Intrinsic Dimension of Data Representations in Deep Neural Networks"
- Kornblith et al. (2019) "Similarity of Neural Network Representations Revisited"

## Why This Matters

Activation geometry measures the manifold structure of internal representations — intrinsic dimensionality, representation similarity, clustering, and curvature. These geometric properties reveal how the model organizes information in its high-dimensional activation space.

**Key references:**
- [Kornblith et al. (2019) "Similarity of Neural Network Representations"](https://arxiv.org/abs/1905.00414) — CKA for comparing representations across models and layers
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.activation_geometry import (
    activation_manifold_dimension,
    representation_similarity_across_inputs,
    activation_cluster_analysis,
    representation_curvature,
    activation_norm_distribution,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)

rng = np.random.RandomState(42)
tokens_list = [jnp.array(rng.randint(0, 100, size=8)) for _ in range(10)]
print(f"Model: {cfg.n_layers} layers, d_model={cfg.d_model}")
print(f"Inputs: {len(tokens_list)} sequences")

In [ ]:
# 1. Activation manifold dimension
dim = activation_manifold_dimension(model, tokens_list)
print(f"Intrinsic dimensionality: {dim['intrinsic_dim']:.2f}")
print(f"Dimensions for 90% variance: {dim['n_for_90pct']}")
print(f"Dimensions for 99% variance: {dim['n_for_99pct']}")
print(f"Top explained variance ratios: {dim['explained_variance'][:5].round(4)}")

In [ ]:
# 2. Representation similarity across inputs
tokens_a = tokens_list[0]
tokens_b = tokens_list[1]
sim = representation_similarity_across_inputs(model, tokens_a, tokens_b)
print(f"Initial similarity: {sim['initial_similarity']:.4f}")
print(f"Final similarity: {sim['final_similarity']:.4f}")
print(f"Divergence layer: {sim['divergence_layer']}")
print(f"Convergence layer: {sim['convergence_layer']}")
print(f"\nLayer-by-layer similarity:")
for l in range(len(sim['layer_similarities'])):
    bar = '#' * int(abs(sim['layer_similarities'][l]) * 30)
    print(f"  Layer {l}: {sim['layer_similarities'][l]:+.4f} {bar}")

In [ ]:
# 3. Activation cluster analysis
clust = activation_cluster_analysis(model, tokens_list, n_clusters=3)
print(f"Cluster assignments: {clust['cluster_assignments']}")
print(f"Cluster sizes: {clust['cluster_sizes']}")
print(f"Within-cluster similarity: {clust['within_cluster_similarity'].round(4)}")
print(f"Between-cluster similarity: {clust['between_cluster_similarity']:.4f}")
print(f"Centroids shape: {clust['cluster_centroids'].shape}")

In [ ]:
# 4. Representation curvature
curv = representation_curvature(model, tokens_list[0])
print(f"Mean curvature: {curv['mean_curvature']:.4f} radians")
print(f"Max curvature at layer: {curv['max_curvature_layer']}")
print(f"Trajectory length: {curv['trajectory_length']:.4f}")
print(f"\nCurvature per layer transition:")
for i, c in enumerate(curv['curvatures']):
    bar = '#' * int(c / 0.1)
    print(f"  Layer {i}->{i+1}->{i+2}: {c:.4f} rad {bar}")

In [ ]:
# 5. Activation norm distribution
norms = activation_norm_distribution(model, tokens_list)
print(f"Mean norm: {norms['mean_norm']:.4f}")
print(f"Std norm: {norms['std_norm']:.4f}")
print(f"Range: [{norms['min_norm']:.4f}, {norms['max_norm']:.4f}]")
print(f"Coefficient of variation: {norms['coefficient_of_variation']:.4f}")
print(f"\nPer-input norms:")
for i, n in enumerate(norms['norms']):
    bar = '#' * int(n * 5)
    print(f"  Input {i}: {n:.4f} {bar}")